# 8. High-Performance Classification with BERT Model (English DistilBERT)

In this notebook, we will go beyond traditional Machine Learning (Logistic Regression, SVM) and standard Deep Learning (TextCNN, FastText) models and use a **Transformer (BERT)** based model, which is the current State-of-the-Art (SOTA) architecture.

**IMPORTANT NOTE:** Since our dataset is massive with 1.6 million reviews, training BERT with all of this data on a standard computer graphics card (GPU) could take days. Therefore:
- We will use the lighter and faster **`distilbert-base-uncased`** model.
- Instead of the entire 1,140,000 training data, we will perform fine-tuning on a **balanced subset of 50,000 or 100,000 reviews**.

If the necessary libraries are not installed, you can run the cell below:

In [10]:
!pip install transformers datasets accelerate evaluate

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [11]:
import os
import pandas as pd
import numpy as np
import torch
import evaluate
import joblib
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

RANDOM_STATE = 42

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in use: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device in use: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## 1. Data Loading and Official Split Reuse

This notebook reuses `features/train_idx.npy`, `features/val_idx.npy`, and `features/test_idx.npy` created in `04_feature_extraction.ipynb`. BERT may still fine-tune on a smaller subset of the official training split for hardware reasons, but validation and final testing stay aligned with the classical models.


In [12]:
print("Loading data and official split indices...")
df = pd.read_csv('data/reviews_preprocessed.csv', usecols=['text', 'label'])
df = df.dropna(subset=['text']).reset_index(drop=True)
df['label_num'] = df['label'].astype(int)

idx_train = np.load('features/train_idx.npy')
idx_val = np.load('features/val_idx.npy')
idx_test = np.load('features/test_idx.npy')

train_source_df = df.iloc[idx_train].reset_index(drop=True)
val_df = df.iloc[idx_val].reset_index(drop=True)
test_df = df.iloc[idx_test].reset_index(drop=True)

# Optional hardware-friendly training subset, sampled only from the official training split.
TRAIN_SUBSET_SIZE = 90000
if len(train_source_df) > TRAIN_SUBSET_SIZE:
    train_df, _ = train_test_split(
        train_source_df,
        train_size=TRAIN_SUBSET_SIZE,
        stratify=train_source_df['label_num'],
        random_state=RANDOM_STATE
    )
    train_df = train_df.reset_index(drop=True)
else:
    train_df = train_source_df

print(f"Training rows used for BERT fine-tuning: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")
print(f"Official test rows: {len(test_df)}")
print("\nTrain class distribution:")
print(train_df['label_num'].value_counts(normalize=True).sort_index())
print("\nTest class distribution:")
print(test_df['label_num'].value_counts(normalize=True).sort_index())


Loading data and official split indices...
Training rows used for BERT fine-tuning: 90000
Validation rows: 244381
Official test rows: 244381

Train class distribution:
label_num
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64

Test class distribution:
label_num
0    0.333336
1    0.333332
2    0.333332
Name: proportion, dtype: float64


In [13]:
# Converting to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df[['text', 'label_num']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text', 'label_num']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'label_num']].reset_index(drop=True))


## 2. Tokenization (Converting Text to a Format BERT Understands)

In [14]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

print("Tokenizing the training set...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing the validation set...")
tokenized_val = val_dataset.map(tokenize_function, batched=True)
print("Tokenizing the official test set...")
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Setting the column name expected by the HuggingFace model to 'labels'
tokenized_train = tokenized_train.rename_column("label_num", "labels")
tokenized_val = tokenized_val.rename_column("label_num", "labels")
tokenized_test = tokenized_test.rename_column("label_num", "labels")

# Converting to PyTorch tensor format
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_val.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])


Tokenizing the training set...


Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Tokenizing the validation set...


Map:   0%|          | 0/244381 [00:00<?, ? examples/s]

Tokenizing the official test set...


Map:   0%|          | 0/244381 [00:00<?, ? examples/s]

## 3. Model Setup and Training (Fine-Tuning)

In [15]:
# evaluate library metrics for calculating Accuracy and F1 Score
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

# Loading the model (3 Classes: Bad, Middle, Good)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [16]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir="./bert_logs",
    logging_steps=500,
    fp16=torch.cuda.is_available(), # If GPU is available, accelerates training with 16-bit precision
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
# Start training
print("Starting BERT Model training...")
trainer.train()

Starting BERT Model training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.477221,0.476870,0.799604,0.799211
2,0.401618,0.483908,0.806356,0.805431
3,0.311511,0.529372,0.803884,0.804395


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=16875, training_loss=0.41255871084707757, metrics={'train_runtime': 2133.0452, 'train_samples_per_second': 126.58, 'train_steps_per_second': 7.911, 'total_flos': 8941708869120000.0, 'train_loss': 0.41255871084707757, 'epoch': 3.0})

## 4. Evaluation and Saving

In [18]:
# Final performance on the official test set shared with the classical models
test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")
print("Official Test Set Results:", test_results)

os.makedirs('results', exist_ok=True)
pd.DataFrame([{
    'Model': 'DistilBERT',
    'Accuracy': test_results.get('test_accuracy'),
    'F1-Macro': test_results.get('test_f1_macro'),
    'Train Rows Used': len(train_df),
    'Validation Rows': len(val_df),
    'Test Rows': len(test_df)
}]).to_csv('results/bert_test_evaluation.csv', index=False)
print("[OK] BERT official test metrics saved to results/bert_test_evaluation.csv")

# Saving the best model
# On Windows, model.safetensors can stay locked when the Flask app has loaded BERT.
# Write PyTorch weights manually to bypass safetensors serialization completely.
bert_save_dir = "models/distilbert_sentiment_model"
os.makedirs(bert_save_dir, exist_ok=True)
model_to_save = trainer.model.module if hasattr(trainer.model, "module") else trainer.model
model_to_save.config.save_pretrained(bert_save_dir)
torch.save(model_to_save.state_dict(), os.path.join(bert_save_dir, "pytorch_model.bin"))
tokenizer.save_pretrained(bert_save_dir)
torch.save(trainer.args, os.path.join(bert_save_dir, "training_args.bin"))
print(f"Model successfully saved to '{bert_save_dir}' folder as pytorch_model.bin!")


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.311511,0.484346,3,0.805369,0.804437


Official Test Set Results: {'test_loss': 0.4843457341194153, 'test_accuracy': 0.805369484534395, 'test_f1_macro': 0.8044366503692482}
[OK] BERT official test metrics saved to results/bert_test_evaluation.csv
Model successfully saved to 'models/distilbert_sentiment_model' folder as pytorch_model.bin!
